# Instasight: Multi‑Agent Instagram Intelligence
## Educational Notebook

This notebook explains the architecture and methodology behind Instasight, a multi‑agent system for Instagram data analysis.

**Key concepts:**
- Data ingestion (CSV or Meta API)
- Normalization to a standard schema
- LLM‑powered analytics (engagement, posting patterns, forecasting)
- BI exports (PowerBI, Tableau)

> **Note:** The production system uses Google ADK for agent orchestration. This notebook simulates the agents using Python functions for clarity.

## 1. Data Ingestion & Normalization

Instagram data can come from two sources:
- **CSV export** from Meta Business Suite
- **Meta Graph API** (live data)

The **ingestion agent** loads the data, and the **normalization agent** maps inconsistent column names to a standard schema.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Simulate a raw Instagram CSV
np.random.seed(42)
n_posts = 50
raw_data = pd.DataFrame({
    'like_count': np.random.randint(10, 1000, n_posts),
    'comment_count': np.random.randint(0, 200, n_posts),
    'saved': np.random.randint(0, 500, n_posts),
    'impressions': np.random.randint(500, 5000, n_posts),
    'reach': np.random.randint(300, 4000, n_posts),
    'timestamp': [datetime.now() - timedelta(days=np.random.randint(0, 30)) for _ in range(n_posts)],
    'media_type': np.random.choice(['IMAGE', 'VIDEO', 'CAROUSEL'], n_posts)
})
print("Raw data columns:", raw_data.columns.tolist())
raw_data.head()

In [ ]:
# Normalization function (as in tools/normalize_instagram_csv.py)
def normalize_instagram_csv(df_raw):
    column_map = {
        "likes": ["like_count", "likes"],
        "comments": ["comment_count", "comments"],
        "saves": ["saved", "save_count"],
        "impressions": ["impressions"],
        "reach": ["reach"],
        "post_timestamp": ["timestamp", "published", "created_time"],
        "content_type": ["media_type", "content_type"],
    }
    df = pd.DataFrame()
    for target, potentials in column_map.items():
        for pot in potentials:
            if pot in df_raw.columns:
                df[target] = df_raw[pot]
                break
        if target not in df:
            if target == "post_timestamp":
                df[target] = pd.NaT
            else:
                df[target] = 0
    df["post_timestamp"] = pd.to_datetime(df["post_timestamp"], errors="coerce")
    df["days_since_post"] = (pd.Timestamp.now() - df["post_timestamp"]).dt.total_seconds() / 86400
    df["content_type"] = df["content_type"].astype(str).str.lower()
    return df

normalized = normalize_instagram_csv(raw_data)
print("Normalized data shape:", normalized.shape)
normalized.head()

## 2. LLM‑Powered Analytics

The **analysis agent** computes:
- Engagement rate = (likes + comments + saves) / reach
- Top performing posts
- Best posting hour and weekday
- Engagement by content type
- Simple forecast (rolling average)

In [ ]:
def engagement_analysis(data):
    data = data.copy()
    data["eng_rate"] = (data["likes"] + data["comments"] + data["saves"]) / data["reach"].replace(0, 1)
    return {
        "average_engagement_rate": data["eng_rate"].mean(),
        "top_posts": data.nlargest(5, "eng_rate")[["post_timestamp", "likes", "comments", "saves", "eng_rate"]].to_dict(orient="records")
    }

def posting_patterns(data):
    data = data.copy()
    data["eng_rate"] = (data["likes"] + data["comments"] + data["saves"]) / data["reach"].replace(0, 1)
    data["hour"] = data["post_timestamp"].dt.hour
    data["weekday"] = data["post_timestamp"].dt.day_name()
    return {
        "best_hour": int(data.groupby("hour")["eng_rate"].mean().idxmax()),
        "best_weekday": data.groupby("weekday")["eng_rate"].mean().idxmax()
    }

def content_clustering(data):
    data = data.copy()
    data["eng_rate"] = (data["likes"] + data["comments"] + data["saves"]) / data["reach"].replace(0, 1)
    return {
        "type_summary": data.groupby("content_type")["eng_rate"].mean().to_dict()
    }

def metrics_forecast(data, window=7):
    data = data.copy()
    data["eng_rate"] = (data["likes"] + data["comments"] + data["saves"]) / data["reach"].replace(0, 1)
    if len(data) < window:
        return {"predicted_eng_rate": data["eng_rate"].mean()}
    avg_growth = data["eng_rate"].rolling(window).mean().iloc[-1]
    return {"predicted_eng_rate": float(avg_growth)}

# Run analysis
analysis = engagement_analysis(normalized)
patterns = posting_patterns(normalized)
clusters = content_clustering(normalized)
forecast = metrics_forecast(normalized)

print("Average engagement rate:", analysis["average_engagement_rate"])
print("Best posting hour:", patterns["best_hour"])
print("Best weekday:", patterns["best_weekday"])
print("Engagement by content type:", clusters["type_summary"])
print("Forecasted engagement rate:", forecast["predicted_eng_rate"])

## 3. Visualization

We can plot engagement trends and top posts.

In [ ]:
# Plot engagement rate over time
normalized["eng_rate"] = (normalized["likes"] + normalized["comments"] + normalized["saves"]) / normalized["reach"].replace(0, 1)
normalized = normalized.sort_values("post_timestamp")
plt.figure(figsize=(12,5))
plt.plot(normalized["post_timestamp"], normalized["eng_rate"], marker='o', linestyle='-', alpha=0.7)
plt.title("Engagement Rate Over Time")
plt.xlabel("Date")
plt.ylabel("Engagement Rate")
plt.grid(True)
plt.show()

# Bar chart for content type engagement
type_eng = clusters["type_summary"]
plt.figure(figsize=(8,4))
plt.bar(type_eng.keys(), type_eng.values(), color=['skyblue', 'salmon', 'lightgreen'])
plt.title("Average Engagement Rate by Content Type")
plt.ylabel("Engagement Rate")
plt.show()

## 4. Exporting for BI Tools

The **export agent** generates a CSV for PowerBI and a placeholder for Tableau Hyper extracts.

In [ ]:
def generate_powerbi_csv(analysis):
    df = pd.DataFrame(analysis["top_posts"])
    # In production, save to file
    return df

bi_df = generate_powerbi_csv(analysis)
print("Top 5 posts for PowerBI:")
bi_df

## 5. Multi‑Agent Orchestration (Simulated)

In the real system, Google ADK coordinates the agents. Here we simulate the pipeline with function calls.

In [ ]:
def run_full_pipeline(file_path):
    # Step 1: Ingestion
    raw = pd.read_csv(file_path) if isinstance(file_path, str) else file_path
    # Step 2: Normalization
    normalized = normalize_instagram_csv(raw)
    # Step 3: Analysis
    analysis = engagement_analysis(normalized)
    patterns = posting_patterns(normalized)
    clusters = content_clustering(normalized)
    forecast = metrics_forecast(normalized)
    # Combine insights
    insights = {
        "analysis": analysis,
        "patterns": patterns,
        "clusters": clusters,
        "forecast": forecast
    }
    # Step 4: Export
    bi_export = generate_powerbi_csv(analysis)
    return insights, bi_export

# Test with a sample CSV
sample_csv = "sample_instagram_data.csv"
raw_data.to_csv(sample_csv, index=False)
insights, bi_export = run_full_pipeline(sample_csv)
print("Pipeline executed successfully.")
print("Top post engagement rate:", insights["analysis"]["top_posts"][0]["eng_rate"])
import os
os.remove(sample_csv)  # clean up

## 6. Business Impact & Next Steps

**Business value:**
- Automates manual data collection and reporting
- Identifies optimal posting times and content types
- Forecasts engagement to plan campaigns
- Exports ready for PowerBI/Tableau dashboards

**Future improvements:**
- Replace simple forecasting with Prophet/ARIMA
- Add sentiment analysis on comments
- Support TikTok, LinkedIn, Twitter
- Deploy as managed agents on Vertex AI

**Testing the real system:**
1. Fix Google ADK imports and API usage.
2. Write unit tests for each tool (use `pytest`).
3. Run the Streamlit app locally with a sample CSV.
4. Set up GitHub Actions for CI/CD.

This notebook demonstrates the core logic. The production system extends it with LLM‑powered agents and scalable orchestration.